# 9.9 最大 softmax 機率很高就可靠嗎？

# 第 9 章：誠實、安全與適當回應

前置：第7章SFT與第8章分項評估。用能驗證真假的小世界教誠實、修正前提與安全邊界，不把課堂成功推廣成任意開放世界安全保證。外部PKU資料的非商用授權與原始安全標籤要保留。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：信心像音量：說得大聲不保證說對**

每箱保留樣本量；文字自述信心是另一回事。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/calibration.svg")))

**先想一想：**把所有logits乘10，知識真的增加了嗎？

softmax只比較目前候選的相對分數。陌生輸入也可能被分得很尖；模型說「我很有信心」又是另一個文字行為。

**把直覺寫成數學：**confidence=max softmax(logits)；它不是由公式保證的正確機率。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
z = torch.tensor([[2.0, 1.0, 0.0]])
for scale in (1.0, 10.0):
    print(scale, (z * scale).softmax(-1).max().item())
print("同一錯誤類別可以更有信心；需要獨立標準答案查證")

**如何讀結果：**只是尺度改變，沒有學新事實；temperature校準也不創造知識。

**只改一件事：**只把最高分的類別設為錯誤標籤，觀察高信心錯答。
